# register-buffer — faded example 3: Save and Restore Buffer Values via state_dict Roundtrip (Faded)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-buffer`. Running the beacon reports progress on the `PyTorch: register_buffer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: register_buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-buffer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-buffer"
DD_SUBTOPIC = "PyTorch: register_buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Because `register_buffer` tensors appear in `state_dict()`, calling `load_state_dict` on a fresh module overwrites its buffer values with the saved ones. This is how running statistics (e.g., batch norm's mean and variance) are preserved across training interruptions. The loaded module's buffer tensor reflects the exact values from the checkpoint.

## Faded exercise 3

A `RunningMax` module tracks the element-wise running maximum in a registered buffer. The module constructor and `update` method are provided. Your task is to **complete the `load_and_verify` function**, which should load a state dict into a fresh module and return whether the buffer value matches the original.

Specifically: create a new `RunningMax(dim)`, call `load_state_dict(sd)` on it, then return `torch.equal(new_module.running_max, original_max_value)`.

**Fill in:** Create a fresh RunningMax(dim), load the state dict into it, and return True if its running_max matches the provided tensor.

In [ ]:
import torch as t
import torch.nn as nn

class RunningMax(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.register_buffer('running_max', t.full((dim,), float('-inf')))

    def update(self, x: t.Tensor) -> None:
        self.running_max = t.maximum(self.running_max, x)

def load_and_verify(dim: int, sd: dict, original_max: t.Tensor) -> bool:
    raise NotImplementedError()  # TODO: Create a fresh RunningMax(dim), load the state dict into it, and return True if its running_max matches the provided tensor.


def _test():
    import torch as t
    t.manual_seed(42)
    dim = 5
    m = RunningMax(dim)
    for _ in range(3):
        m.update(t.randn(dim))
    sd = m.state_dict()
    original_max = m.running_max.clone()
    result = load_and_verify(dim, sd, original_max)
    assert result is True, f'Expected True, got {result}'
    # Also verify buffer is actually in state_dict
    assert 'running_max' in sd, 'running_max missing from state_dict'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class RunningMax(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.register_buffer('running_max', t.full((dim,), float('-inf')))

    def update(self, x: t.Tensor) -> None:
        self.running_max = t.maximum(self.running_max, x)

def load_and_verify(dim: int, sd: dict, original_max: t.Tensor) -> bool:
    new_module = RunningMax(dim)
    new_module.load_state_dict(sd)
    return t.equal(new_module.running_max, original_max)
```
</details>